In [1]:
!pip install rank-bm25

In [2]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi

In [3]:
df_q = pd.read_csv('acgs_d.csv')

In [4]:
companies = [
    "PT AKR Corporindo Tbk",
    "PT Bank CIMB Niaga Tbk",
    "PT Bank Central Asia Tbk",
    "PT Bank Jago Tbk",
    "PT Bank Mandiri Tbk",
    "PT Bank Negara Indonesia Tbk",
    "PT Bank Rakyat Indonesia Tbk",
    "PT Indosat Tbk",
    "PT Telkom Indonesia Tbk",
    "PT Wijaya Karya Tbk",
]

In [5]:
company_offsets = {
    "PT AKR Corporindo Tbk": 2,
    "PT Bank CIMB Niaga Tbk": -2,
    "PT Bank Central Asia Tbk": 2,
    "PT Bank Jago Tbk": 2,
    "PT Bank Mandiri Tbk": 2,
    "PT Bank Negara Indonesia Tbk": 2,
    "PT Bank Rakyat Indonesia Tbk": 2,
    "PT Indosat Tbk": 2,
    "PT Telkom Indonesia Tbk": 2,
    "PT Wijaya Karya Tbk": 0,
}

In [6]:
topks = [5, 10, 20]

# Helper Function



> Preprocessing



In [7]:
def preprocessing(text):
    text = text.lower()
    text = text.split()

    return text



> Querying



In [8]:
def match_text(bm25, queries, topk):
    topk_bm25_res_lod = []

    for q_idx, query in enumerate(queries):
        scores = bm25.get_scores(query)

        sorted_indices = np.argsort(scores)[::-1]

        ranking = sorted_indices[:topk]

        for rank, doc_idx in enumerate(ranking, start=1):
            topk_bm25_res_lod.append({
                "q_idx": q_idx,
                "rank": rank,
                "d_idx": int(doc_idx)
            })

    return pd.DataFrame(topk_bm25_res_lod)

In [9]:
def map_to_page_level(bm25, queries, pages):
    page_level_res = []

    for i, query in enumerate(queries):
        scores = bm25.get_scores(query)

        for j, score in enumerate(scores):
            page_level_res.append({
                "q_idx": i,
                "d_idx": j,
                "page_retrieved": pages[j],
                "score": score
            })

    return pd.DataFrame(page_level_res)

In [10]:
def max_aggregate(page_level_ret_res, topk):

    idx = (
        page_level_ret_res
        .groupby(["q_idx", "page_retrieved"])["score"]
        .idxmax()
    )

    page_level_ret_res = page_level_ret_res.loc[idx].reset_index(drop=True)

    page_level_ret_res = page_level_ret_res.sort_values(
        ["q_idx", "score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    aggregated_ret_res = (
        page_level_ret_res
        .groupby("q_idx")
        .head(topk)
        .reset_index(drop=True)
    )

    aggregated_ret_res["rank"] = (
        aggregated_ret_res.groupby("q_idx").cumcount() + 1
    )

    return aggregated_ret_res



> Evaluation



In [11]:
def parse_anchor_pages(s, offset=0):
    pages = set()

    if pd.isna(s):
        return pages

    s = str(s).replace('"', '').replace("'", "").strip()
    if s == "":
        return pages

    for part in s.split(";"):
        part = part.strip()
        if part == "":
            continue

        if "-" in part:
            a, b = part.split("-", 1)
            a, b = a.strip(), b.strip()
            if a == "" or b == "":
                continue
            pages.update(range(int(a) + offset, int(b) + offset + 1))
        else:
            pages.add(int(part) + offset)

    return pages

In [12]:
def merge_parse_df(topk_df, query_df, corpus_df, evaluator_df, offset):
    df_merged = (
        topk_df
        .merge(query_df, left_on='q_idx', right_index=True, how='left')
        .merge(corpus_df, left_on='d_idx', right_index=True, how='left')
        .merge(evaluator_df, left_on='q_idx', right_index=True, how='left')
    )

    df_merged["anchor_set"] = df_merged["anchor_pages"].apply(
        lambda x: parse_anchor_pages(x, offset=offset)
    )

    return df_merged

In [13]:
def filter_df(df):
    df_final = df[(df["answerable"] == True) & (df["in_scope"] == True)]
    df_final = df_final.copy()
    df_final["is_hit"] = df_final.apply(lambda r: r["page"] in r["anchor_set"], axis=1)

    return df_final

In [14]:
def generate_score(df, company, topk):

    hit_score = (
        df[df["rank"] <= topk]
        .groupby("qid")["is_hit"]
        .any()
        .mean()
    )

    df_k = df[df["rank"] <= topk].copy()

    hits = df_k[df_k["is_hit"]]

    first_hit_rank = (
        hits
        .sort_values(["qid", "rank"])
        .groupby("qid")["rank"]
        .first()
    )

    reciprocal = 1 / first_hit_rank

    mrr_score = (
        reciprocal
        .reindex(df["qid"].unique(), fill_value=0)
        .mean()
    )

    return {
        "company": company,
        "k": int(topk),
        "hit_at_k": float(hit_score),
        "mrr_at_k": float(mrr_score)
    }

# Main Loop

In [15]:
queries = df_q['q_text'].tolist()
preprocessed_queries = [preprocessing(query) for query in queries]

bm25_eval = []

candidate_pool = 100

for company in companies:
    corpus = pd.read_excel(f"{company}.xlsx")
    documents = corpus["d_text"].tolist()
    pages = corpus["page"].tolist()

    evaluator = pd.read_excel(f"{company}_Evaluator.xlsx")

    offset = company_offsets.get(company, 0)

    preprocessed_docs = [preprocessing(doc) for doc in documents]

    # indexing
    bm25 = BM25Okapi(preprocessed_docs)

    page_level_res = map_to_page_level(bm25, preprocessed_queries, pages)

    page_level_res_top = (
        page_level_res
        .sort_values(["q_idx", "score"], ascending=[True, False])
        .groupby("q_idx")
        .head(candidate_pool)
        .reset_index(drop=True)
    )

    for topk in topks:
        # querying
        aggregated_ret_res = max_aggregate(page_level_res_top, topk)

        # eval
        merged_df = merge_parse_df(aggregated_ret_res, df_q, corpus, evaluator, offset)
        final_df = filter_df(merged_df)

        bm25_score = generate_score(final_df, company, topk)

        bm25_eval.append(bm25_score)

In [16]:
bm25_eval = pd.DataFrame(bm25_eval)

In [17]:
bm25_eval.head()

,company,k,hit_at_k,mrr_at_k
0,PT AKR Corporindo Tbk,5,0.392157,0.207843
1,PT AKR Corporindo Tbk,10,0.568627,0.230851
2,PT AKR Corporindo Tbk,20,0.686275,0.240118
3,PT Bank CIMB Niaga Tbk,5,0.612903,0.495699
4,PT Bank CIMB Niaga Tbk,10,0.645161,0.499731


In [18]:
bm25_eval.count()

,0
company,30
k,30
hit_at_k,30
mrr_at_k,30


In [19]:
bm25_eval.to_csv("bm25_eval.csv", index=False)